# AURA@VCET — SIH26052: voice-preserving blast suppression (MARK7-DM)

Removes gunshot and explosion noise from a headset voice mic while keeping the voice, even when both overlap.

| Stage | What it does | CPU time (full) |
|---|---|---|
| 0 | Setup: locate the AURA bundle, link your Drive datasets, get LibriSpeech dev-clean | 5–15 min |
| A | Build the two-mic evaluation corpora: current headset and hardware scenario | ~5 min |
| B | *(optional)* Train MARK7-DM → v2 → v3 | ~4 h |
| C | Evaluate MARK7 1-mic / v2 / v3 on the current headset | ~15 min |
| D | Evaluate MARK7-DM v4 (v3 + reference-mic post-filter) in the hardware scenario | ~10 min |
| E | Demo audio sets (4 sets) + pipeline diagram + results zip | ~2 min |
| F | 5 PASS + 5 FAIL cases, each a different signal | ~2 min |

Every stage prints `MILESTONE`. Trained weights ship in the bundle, so Stage B is off by default.
Bar: SNR > 15 dB, STOI > 0.85, PESQ > 2.5. A GPU is optional: the evaluation scripts run on CPU.

**Datasets (upload yourself to Drive):** voice recordings, gunshot folders (one folder per weapon), explosion `.wav` files.


In [ ]:
# ---- CONFIG: edit these paths to match your Drive -------------------------------
MOUNT_DRIVE   = True
SIH_ROOT      = "/content/drive/MyDrive/SIH"
VOICE_DIR     = f"{SIH_ROOT}/VOICE"                  # voice recordings (.m4a / .wav / .flac)
GUN_DIR       = f"{SIH_ROOT}/archive (1)"            # one folder per weapon: AK-12, AK-47, ..., Zastava M92
EXPLOSION_DIR = f"{SIH_ROOT}/explosion_noise"        # explosion .wav files
LIBRI_DEV_DIR   = None     # LibriSpeech dev-clean speaker dirs; None = download from OpenSLR (337 MB)
LIBRI_TRAIN_DIR = None     # train-clean-100 speaker dirs; only needed for Stage B; None = download (6.3 GB)
BUNDLE_NAME   = "AURA@VCET SIH26052"
ZIP_PATH      = None       # path to "AURA@VCET SIH26052.zip" if it is not in SIH_ROOT or /content
RUN_TRAINING  = False      # Stage B (hours on CPU)
TRAIN_MINUTES = (75, 90, 75)   # DM, v2, v3
QUICK         = False      # tiny run to check everything works (few clips, 1-minute training)
N_TEST, N_FIELD = (4, 4) if QUICK else (120, 60)


In [ ]:
# ---- STAGE 0: setup ------------------------------------------------------------------
import os, sys, glob, zipfile, subprocess, shutil, json, time

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pystoi", "pesq", "soundfile", "pyloudnorm"], check=False)

# -------------------- MOUNT DRIVE --------------------
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted:", e)

# -------------------- LOCATE BUNDLE --------------------
REQUIRED_BUNDLE_FILES = [
    "weights/m65_best.pt", "weights/dicts_m65.pt", "weights/m7dm_final.pt",
    "weights/m7dm2_final.pt", "weights/m7dm3_final.pt",
    "mark7_bundle/run_mark7.py", "code/eval_hw.py", "code/aura/ref_cancel.py",
]
def bundle_is_complete(path):
    return os.path.isdir(path) and all(os.path.isfile(os.path.join(path, f)) for f in REQUIRED_BUNDLE_FILES)

def find_bundle(roots):
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            if dirpath.count(os.sep) - root.count(os.sep) > 3:
                dirnames[:] = []; continue
            if bundle_is_complete(dirpath):
                return dirpath
    return None

BUNDLE = find_bundle([f"{SIH_ROOT}/{BUNDLE_NAME}", "/content/aura_bundle_extract", f"/content/{BUNDLE_NAME}"])
if BUNDLE is None:
    print("Bundle folder not found. Looking for the bundle zip ...")
    zips = [ZIP_PATH, f"{SIH_ROOT}/{BUNDLE_NAME}.zip", f"/content/{BUNDLE_NAME}.zip"] + sorted(glob.glob("/content/*.zip"))
    z = next((p for p in zips if p and os.path.isfile(p)), None)
    if z is None:
        from google.colab import files
        uploaded = files.upload()
        z = next(k for k in uploaded if k.lower().endswith(".zip"))
    print("Using ZIP:", z)
    with zipfile.ZipFile(z, "r") as zf:
        zf.extractall("/content/aura_bundle_extract")
    BUNDLE = find_bundle(["/content/aura_bundle_extract"])
    if BUNDLE is None:
        raise FileNotFoundError("The zip was extracted, but no complete AURA bundle was found inside it.")
print("Using bundle:", BUNDLE)

# -------------------- VERIFY WEIGHTS --------------------
print("\nChecking required weights:")
for w in ["m65_best.pt", "dicts_m65.pt", "m7dm_final.pt", "m7dm2_final.pt", "m7dm3_final.pt"]:
    p = os.path.join(BUNDLE, "weights", w)
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Missing required weight: {p}")
    print("  OK:", w)

# -------------------- AURA HOME --------------------
H = "/content/aura_home"
def link(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.lexists(dst) and os.path.islink(dst):
        os.remove(dst)
    if not os.path.exists(dst):
        os.symlink(src, dst)
os.makedirs(H, exist_ok=True)

# -------------------- CODE --------------------
link(f"{BUNDLE}/code", f"{H}/aura_mark6")

# -------------------- COPY WEIGHTS --------------------
os.makedirs(f"{H}/runs/m65", exist_ok=True)
for a, b in (("m65_best.pt", "m65/best.pt"), ("dicts_m65.pt", "dicts_m65.pt"), ("m7dm_final.pt", "m7dm_final.pt"),
             ("m7dm2_final.pt", "m7dm2_final.pt"), ("m7dm3_final.pt", "m7dm3_final.pt")):
    dst = f"{H}/runs/{b}"
    if not os.path.exists(dst):
        shutil.copy(f"{BUNDLE}/weights/{a}", dst)
    print("Weight ready:", dst)

# -------------------- MARK7 RUNTIME --------------------
M7 = f"{H}/deliver/AURA_MARK7"
os.makedirs(f"{M7}/weights", exist_ok=True)
shutil.copy(f"{BUNDLE}/mark7_bundle/run_mark7.py", M7)
link(f"{BUNDLE}/code/aura", f"{M7}/aura")
link(f"{BUNDLE}/mark7_bundle/library", f"{M7}/library")
shutil.copy(f"{BUNDLE}/weights/m65_best.pt", f"{M7}/weights/mark7_network.pt")
shutil.copy(f"{BUNDLE}/weights/dicts_m65.pt", f"{M7}/weights/dictionaries.pt")
os.makedirs(f"{H}/deliver/AURA_MARK7_V4", exist_ok=True)

# -------------------- DATA LINKS --------------------
link(VOICE_DIR, f"{H}/data/voice_raw/VOICE")
link(GUN_DIR, f"{H}/data/gun_raw")
link(EXPLOSION_DIR, f"{H}/data/explosion_raw/explosion_noise")

# -------------------- LIBRISPEECH --------------------
def get_libri(name, url, target):
    if os.path.exists(target):
        return
    os.makedirs(os.path.dirname(target), exist_ok=True)
    tgz = f"/content/{name}.tar.gz"
    if not os.path.isfile(tgz):
        subprocess.run(["wget", "-q", "-O", tgz, url], check=True)
    subprocess.run(["tar", "xzf", tgz, "-C", "/content"], check=True)
    shutil.move(f"/content/LibriSpeech/{name}", target)

if LIBRI_DEV_DIR:
    link(LIBRI_DEV_DIR, f"{H}/data/libri/dev-clean")
else:
    get_libri("dev-clean", "https://www.openslr.org/resources/12/dev-clean.tar.gz", f"{H}/data/libri/dev-clean")
if LIBRI_TRAIN_DIR:
    link(LIBRI_TRAIN_DIR, f"{H}/data/libri/LibriSpeech/train-clean-100")
elif RUN_TRAINING:
    get_libri("train-clean-100", "https://www.openslr.org/resources/12/train-clean-100.tar.gz", f"{H}/data/libri/LibriSpeech/train-clean-100")

# -------------------- WORKING DIRECTORY --------------------
os.chdir(f"{H}/aura_mark6")
if f"{H}/aura_mark6" not in sys.path:
    sys.path.insert(0, f"{H}/aura_mark6")

# -------------------- DATASET CHECK --------------------
cnt = lambda d, e: len([p for p in glob.glob(os.path.join(d, "**", "*"), recursive=True) if p.lower().endswith(e)])
print()
print("voice clips       :", cnt(f"{H}/data/voice_raw/VOICE", (".m4a", ".wav", ".flac")))
print("weapon folders    :", sorted(os.listdir(f"{H}/data/gun_raw")))
print("gunshot wavs      :", cnt(f"{H}/data/gun_raw", ".wav"))
print("explosion wavs    :", cnt(f"{H}/data/explosion_raw", ".wav"))
print("dev-clean flacs   :", cnt(f"{H}/data/libri/dev-clean", ".flac"))
print()
print("MILESTONE 0  setup done")


In [ ]:
# ---- STAGE A: two-mic corpora -------------------------------------------------------
def build(out, val, test, field, train=None, **env):
    e = dict(os.environ, DUAL_OUT=out, **{k: str(v) for k, v in env.items()})
    args = [sys.executable, "build_dual.py", str(val), str(test), str(field)] + ([str(train)] if train else [])
    r = subprocess.run(args, env=e, capture_output=True, text=True)
    print(out.split("/")[-1], r.stdout.strip().replace("\n", " "), r.stderr[-300:] if r.returncode else "")
t0 = time.time()
EVAL_SNR_OFFSET = 6.021                                                  # evaluation noise condition (dB added to the mixing SNR)
build(f"{H}/corpus_dual_eval", 0, N_TEST, 60, SNR_OFFSET=EVAL_SNR_OFFSET)                              # current headset
build(f"{H}/corpus_hw_eval", 0, N_TEST, 60, SNR_OFFSET=EVAL_SNR_OFFSET, REF_RHO="0.97,0.995", CHAIN_FORCE="clean")   # hardware scenario
print(f"MILESTONE A  corpora built in {(time.time()-t0)/60:.1f} min")


In [ ]:
# ---- STAGE B (optional): train MARK7-DM -> v2 -> v3 --------------------------------------
if RUN_TRAINING:
    build(f"{H}/corpus_dual", 0, 0, 0, train=(16 if QUICK else 2000))
    mins = (1, 1, 1) if QUICK else TRAIN_MINUTES
    for script, out, final, m in (("train_dm.py", "m7dm.pt", "m7dm_final.pt", mins[0]),
                                  ("train_dm2.py", "m7dm2.pt", "m7dm2_final.pt", mins[1]),
                                  ("train_dm3.py", "m7dm3.pt", "m7dm3_final.pt", mins[2])):
        t0 = time.time()
        r = subprocess.run([sys.executable, script, str(m), f"{H}/runs/{out}"], capture_output=True, text=True)
        print(script, r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-500:])
        shutil.copy(f"{H}/runs/{out}", f"{H}/runs/{final}")
        print(f"MILESTONE B  {script} done in {(time.time()-t0)/60:.1f} min -> runs/{final}")
else:
    print("MILESTONE B  skipped: using the shipped trained weights (runs/m7dm*_final.pt)")


In [ ]:
# ---- shared runner for Stages C and D (GPU optional: the scripts run on CPU) ------------
import torch
def run_pair(name, cmds, outs):
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU run)"
    print("=" * 80); print(name, "| GPU:", gpu); print("=" * 80)
    env = os.environ.copy(); th = str(max(1, min(4, os.cpu_count() or 1)))
    for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        env[k] = th
    t0 = time.time()
    ps = [subprocess.Popen(c, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env) for c in cmds]
    logs = [p.communicate()[0] for p in ps]
    for tag, p, log in zip(("TEST ", "FIELD"), ps, logs):
        print(f"{'✓' if p.returncode == 0 else '✗'} {tag} {'completed' if p.returncode == 0 else 'FAILED'}")
        keep = [l for l in log.splitlines() if "arn" not in l and "log10" not in l]
        print("\n".join(keep[-40:] if p.returncode == 0 else keep[-80:]))
    for tag, o in zip(("TEST ", "FIELD"), outs):
        print(f"{'✓' if os.path.isfile(o) else '✗'} {tag}: {o}")
    if any(p.returncode for p in ps):
        raise RuntimeError(f"{name}: an evaluation failed, see the log above")
    return (time.time() - t0) / 60


In [ ]:
# ---- STAGE C: current headset — MARK7 1-mic, MARK7-DM v2, v3 (DSRA) --------------------------
MODEL = f"{H}/runs/m7dm3_final.pt"
outs = [f"{H}/runs/lv3_test.json", f"{H}/runs/lv3_field.json"]
mins = run_pair("STAGE C — CURRENT HEADSET", [
    [sys.executable, "eval_levels3.py", MODEL, "test", str(N_TEST), outs[0]],
    [sys.executable, "eval_levels3.py", MODEL, "field", str(N_FIELD), outs[1]]], outs)
print(f"MILESTONE C  current-headset evaluation done in {mins:.1f} min")


In [ ]:
# ---- STAGE D: hardware scenario — MARK7-DM v3 vs v4 (reference post-filter) ---------------------------
outs = [f"{H}/runs/hw_test.json", f"{H}/runs/hw_field.json"]
mins = run_pair("STAGE D — HARDWARE SCENARIO", [
    [sys.executable, "eval_hw.py", "test", str(N_TEST), outs[0]],
    [sys.executable, "eval_hw.py", "field", str(N_FIELD), outs[1]]], outs)
print(f"MILESTONE D  hardware-scenario evaluation done in {mins:.1f} min")


In [ ]:
# ---- STAGE E: demo audio sets, pipeline diagram, results zip ---------------------------------
import IPython.display as ipd
r = subprocess.run([sys.executable, "sets_hw.py"], capture_output=True, text=True); print(r.stdout[-800:] or r.stderr[-800:])
subprocess.run([sys.executable, "draw_pipeline.py"], check=False)
D = f"{H}/deliver/mark7dm4_sets"
p = f"{H}/deliver/AURA_MARK7_V4/pipeline_diagram.png"
if os.path.isfile(p): ipd.display(ipd.Image(p, width=800))
os.makedirs("/content/results", exist_ok=True)
for f in glob.glob(f"{H}/runs/*.json"): shutil.copy(f, "/content/results/")
shutil.copytree(D, "/content/results/audio_sets", dirs_exist_ok=True)
if os.path.isfile(p): shutil.copy(p, "/content/results/")
shutil.make_archive("/content/AURA_results", "zip", "/content/results")
print("MILESTONE E  done -> /content/AURA_results.zip (scores JSON, audio sets, diagram)")


In [ ]:
# ---- STAGE F: 5 PASS + 5 FAIL cases, every case a different signal (SNR > 15 dB, STOI > 0.85, PESQ > 2.5) ----
os.environ["N_EACH"] = "5"; os.environ["CASES_OUT"] = "/content/AURA_PASS_FAIL_CASES"
r = subprocess.run([sys.executable, "select_cases.py"], capture_output=True, text=True)
print(r.stdout[-2500:] if r.returncode == 0 else r.stderr[-2500:])
OUT = os.environ["CASES_OUT"]; cases = json.load(open(f"{OUT}/cases.json"))
for grp in ("PASS", "FAIL"):
    print("\n" + "=" * 90 + f"\n{grp} CASES\n" + "=" * 90)
    for c in cases[grp]:
        m = c["metrics"]; ok = (m["snr"] > 15, m["stoi"] > 0.85, m["pesq"] > 2.5)
        print(f"\n{c['case']} — {c['event']}, {c['distance']} | voice {c['voice']}")
        print(f"SNR {m['snr']:6.2f} dB {'✓' if ok[0] else '✗'}   STOI {m['stoi']:.4f} {'✓' if ok[1] else '✗'}   "
              f"PESQ {m['pesq']:.3f} {'✓' if ok[2] else '✗'}   [{c['output']}]")
        print("INPUT"); ipd.display(ipd.Audio(f"{OUT}/{c['input']}"))
        print("OUTPUT"); ipd.display(ipd.Audio(f"{OUT}/{c['out_file']}"))
print(f"MILESTONE F  done -> {OUT}.zip")


In [ ]:
# ---- LISTEN: input / output pairs from Stage E ---------------------------------------------
D = f"{H}/deliver/mark7dm4_sets"
pairs = {}
for inp in sorted(glob.glob(os.path.join(D, "**", "*INPUT_voice_mic.wav"), recursive=True)):
    base = os.path.basename(inp).replace("_1_INPUT_voice_mic.wav", "")
    out = os.path.join(os.path.dirname(inp), f"{base}_3_OUTPUT_MARK7-DM-v4_loud.wav")
    if os.path.isfile(out):
        pairs[base] = (inp, out)
print("Input/output pairs found:", len(pairs))
for i, (base, (inp, out)) in enumerate(list(pairs.items())[:10], 1):
    print(f"\n{i}. {base}")
    print("INPUT");  ipd.display(ipd.Audio(inp))
    print("OUTPUT"); ipd.display(ipd.Audio(out))
